# Rail Detection Deep Learning Project

In [ ]:
# importing packages
import csv
import pandas as pd 
import os 
import shutil
import tensorflow as tf 
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.model_selection import train_test_split


## Reading and cleaning the dataset
Using pandas and shutil to make a clean train, validation and test split. Images with two labels are removed, but the non-conflicting augmented images are kept so the dataset does not become too small.


In [ ]:
df = pd.read_csv('train _combined.csv')
df.head()

# Folders
raw_directory = 'raw_data'
split_directory = 'data_clean_split'

# These images have both labels in the csv, so they are skipped
label_count = df.groupby('images')['labels'].nunique()
problem_images = label_count[label_count > 1].index

clean_df = df[~df['images'].isin(problem_images)].copy()

# Keep one row per image
clean_df = clean_df.drop_duplicates(subset='images')

print("Images before cleaning:", len(df))
print("Removed label conflicts:", len(problem_images))
print("Clean images:", len(clean_df))
print(clean_df['labels'].value_counts())

# Make train, validation and test splits
train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df['labels'],
    random_state=1337
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['labels'],
    random_state=1337
)

# Start fresh each time this cell is run
if os.path.exists(split_directory):
    shutil.rmtree(split_directory)

for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    for index, row in split_df.iterrows():
        file_name = row['images']
        label = row['labels']
        class_folder = 'defective' if label == 'Defective' else 'non_defective'
        destination_folder = os.path.join(split_directory, split_name, class_folder)
        os.makedirs(destination_folder, exist_ok=True)
        shutil.copy(
            os.path.join(raw_directory, file_name),
            os.path.join(destination_folder, file_name)
        )

print("Done with the clean split")
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))


## Konstanter

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 1337

keras.utils.set_random_seed(SEED)


In [ ]:
def defective_recall(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)

    predicted_class = tf.cast(y_pred >= 0.5, tf.float32)

    true_defective = 1 - y_true
    predicted_defective = 1 - predicted_class

    correct_defective = tf.reduce_sum(true_defective * predicted_defective)
    missed_defective = tf.reduce_sum(true_defective * (1 - predicted_defective))

    return correct_defective / (correct_defective + missed_defective + keras.backend.epsilon())


## Preprocessing 
Vi kommer inte få flera bilder direkt nu, eftersom hela processen körs "live" dvs att vi kommer lägga in hela data_augmentation som första lagret i modellen. PÅ det sättet kommer effekterna att appliceras live medan bilderna laddas upp till nätverket.

In [ ]:
# Data Augmentation 
data_augmentation = keras.Sequential(
    [
         
        layers.RandomFlip("horizontal"), 
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1), 
        layers.RandomBrightness(0.1),
        layers.Rescaling(1./255)
    ], 
    name = "data_augmentation",
)

train_ds = tf.keras.utils.image_dataset_from_directory(
    "data_clean_split/train",
    seed = SEED,
    shuffle = True,
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data_clean_split/val",
    shuffle = False,
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data_clean_split/test",
    shuffle = False,
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE,
)

class_names = train_ds.class_names
print(class_names)


# Följande koden används för att visualisera bilderna efter data Augmentation för att se till att allting ser bra ut oc inga bilder förstörs

In [ ]:


# Plocka ut EN batch (32 bilder) från träningsdatan
for images, lbls in train_ds.take(1):
    frst_img = images[0]
    
    # För att få plats med två bilder bredvid varandra och jämföra dessa med varandra 
    plt.figure(figsize=(12, 6))
    
    # Plats för orginal bilden till vänster
    plt.subplot(1, 2, 1) 
    
    # Omvandlar 0-255 till 0-1 floats för plt 
    original_to_plot = frst_img.numpy() / 255.0
    original_to_plot = np.clip(original_to_plot, 0, 1)
    
    plt.imshow(original_to_plot)
    plt.title("Före: Original")
    plt.axis("off")
    
    
    # Place for the Augmentated pictures on the right side
    plt.subplot(1, 2, 2) # 1 rad, 2 kolumner, bild 2
    
    # Skickar bilden genom första layer
    augmented_image = data_augmentation(tf.expand_dims(frst_img, 0), training=True)
    
    # Normalisera bilden 
    aug_to_plot = augmented_image[0].numpy()
    aug_to_plot = np.clip(aug_to_plot, 0, 1)
    
    plt.imshow(aug_to_plot)
    plt.title("Efter: Augmenterad")
    plt.axis("off")
    
    # Sparar bilden till rapporten 
    plt.savefig('before_and_after_augmentation.png', bbox_inches='tight')
    plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size = AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size = AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size = AUTOTUNE)


# Now we create the model 
build the simple baseline CNN from scratch.

In [ ]:
# we use the sequential type because the model is a simple layer-by-layer pipeline.
model = keras.Sequential([
    
    #This is the layer that applies augmentation during training.
    data_augmentation,

    #Then comes the first convolutional layer:
        #Conv2D learns small visual patterns, such as edges, lines, cracks and texture.
    layers.Conv2D(32, 3, activation="relu"),

    #MaxPooling reduces the images size.
    layers.MaxPooling2D(),

    #Second convolution layer. 
        #This learns more complex patterns based on the first layer's simple patterns.
    layers.Conv2D(64, 3, activation="relu"),

    #MaxPooling reduces the images size again.
    layers.MaxPooling2D(),

    # This learns even higher-level patterns, such as damaged rail shapes or larger textures.
    layers.Conv2D(128, 3, activation="relu"),

    # Reduce size again before sending features to the classifier part.
    layers.MaxPooling2D(),

    # Instead of flattening every pixel location into a huge vector,
    # this averages each feature map into one number.
    layers.GlobalAveragePooling2D(),


    # This combines the learned image features to make a classification decision.
    layers.Dense(64, activation="relu"),


    # Now using "Dropout" the model drops out randomly 30% of neurons during training to help prevent the model from memorizing the training images 

    layers.Dropout(0.3),

    # We use 1 neuron because this is binary classification.
    # Sigmoid outputs a probability between 0 and 1.
    # Example: 0.90 means the model strongly predicts class 1.
    layers.Dense(1, activation="sigmoid")
])

In [ ]:
# This helps you check the number of layers, output shapes, and trainable parameters.
model.summary()

# Next step is to compile the model 

In [ ]:
model.compile(

    #First we choose the optimizer Adam 
    optimizer="adam",

    #We choose the loss function 
    loss="binary_crossentropy",

    metrics=[

        "accuracy",

        keras.metrics.Precision(name="precision"),

        keras.metrics.Recall(name="recall"),

        defective_recall
    ]
)

In [ ]:
model.summary()

# Training the model now 


In [ ]:
EPOCHS = 30


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# Record accuracy, loss, precision, recall, and F1-score


In [ ]:

# Convert the training history into a table.

history_df = pd.DataFrame(history.history)

# F1-score combines precision and recall into one value.
# Formula: F1 = 2 * (precision * recall) / (precision + recall)
# We add a very small number (1e-7) to avoid division by zero.
history_df["f1_score"] = 2 * (
    history_df["precision"] * history_df["recall"]
) / (history_df["precision"] + history_df["recall"] + 1e-7)

# Validation F1-score uses validation precision and validation recall.
# This is important because validation results better show performance on unseen data.
history_df["val_f1_score"] = 2 * (
    history_df["val_precision"] * history_df["val_recall"]
) / (history_df["val_precision"] + history_df["val_recall"] + 1e-7)

# Add an epoch number column so the table is easier to read.
history_df["epoch"] = range(1, len(history_df) + 1)

# Choose the columns we want to record for the project report.
metrics_table = history_df[[
    "epoch",
    "accuracy",
    "loss",
    "precision",
    "recall",
    "defective_recall",
    "f1_score",
    "val_accuracy",
    "val_loss",
    "val_precision",
    "val_recall",
    "val_defective_recall",
    "val_f1_score"
]]

# Save the metrics table as a CSV file.
# This gives you a permanent record of the baseline CNN results.
metrics_table.to_csv("baseline_cnn_metrics_fixed.csv", index=False)

# Show the table inside the notebook.
metrics_table

# Save plots for training/validation accuracy and loss

In [ ]:


# Create one figure with two plots side by side.
plt.figure(figsize=(12, 5))
epochs = range(1, len(history.history["accuracy"]) + 1)

# Plot 1: training accuracy vs validation accuracy.
# If training accuracy is much higher than validation accuracy, the model may be overfitting.
plt.subplot(1, 2, 1)
plt.plot(epochs, history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs, history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline CNN Accuracy")
plt.legend()

# Plot 2: training loss vs validation loss.
# Loss tells us how wrong the model is; lower loss is better.
# If validation loss rises while training loss falls, that is another sign of overfitting.
plt.subplot(1, 2, 2)
plt.plot(epochs, history.history["loss"], marker="o", label="Training loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline CNN Loss")
plt.legend()

# tight_layout makes sure labels and titles do not overlap.
plt.tight_layout()

# Save the plot for the report.
plt.savefig("baseline_cnn_accuracy_loss_fixed.png", dpi=300, bbox_inches="tight")

# Display the plot in the notebook.
plt.show()

# Print the final recorded results

In [ ]:
# Get the last row from the metrics table.
# The last row represents the final epoch of training.
final_results = metrics_table.iloc[-1]

# Print the final training and validation metrics in a readable format.
print("Final baseline CNN results")
print("Training accuracy:", final_results["accuracy"])
print("Training loss:", final_results["loss"])
print("Training precision:", final_results["precision"])
print("Training recall:", final_results["recall"])
print("Training F1-score:", final_results["f1_score"])
print("Training defective recall:", final_results["defective_recall"])
print("Validation accuracy:", final_results["val_accuracy"])
print("Validation loss:", final_results["val_loss"])
print("Validation precision:", final_results["val_precision"])
print("Validation recall:", final_results["val_recall"])
print("Validation F1-score:", final_results["val_f1_score"])
print("Validation defective recall:", final_results["val_defective_recall"])

# Evaluation of the models performance per class 
we collect the correct labels from the validation dataset and the predited labels from the model history to create a confusion matrix

In [ ]:
# y_true will store the true labels from the test data 
y_true = []

# y_pred will store the labels predicted by the trainde model 
y_pred = []

for images, labels in test_ds:

    probabilities = model.predict(images)

    #convert predicted probabilites into 1D array with class labels 1 and 0 
    predicted_labels = (probabilities >= 0.5).astype("int32").flatten()

    #adding the real labels to y_true.
    y_true.extend(labels.numpy())

    #adding the predicted labels to y_pred.
    y_pred.extend(predicted_labels)


#Convert lists into numpy arrays so that sklearn can work with them easier 
y_true = np.array(y_true)
y_pred = np.array(y_pred)


In [ ]:
# This confirms that we have one prediction for each real label.
print("Class names:", class_names)
print("Number of real labels:", len(y_true))
print("Number of predicted labels:", len(y_pred))

# Confusion matrix 

In [ ]:
#using the sklearn built in confusion matrix 
cm = confusion_matrix(y_true, y_pred)

# Create the classification report.
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
)

In [ ]:
print("Class names:", class_names)

# Print the confusion matrix.
print(cm)

# Print the classification report.
print(report)

The baseline CNN trained from scratch showed limited ability to detect defective rails.
Although it classified non-defective images well, it missed many defective examples.
This suggests that the model did not learn sufficiently strong defect-related features from the available dataset.
Therefore, transfer learning with a pretrained CNN such as ResNet50 is expected to improve performance.


# We now save the evaluation results 

In [ ]:
with open("baseline_cnn_classification_report_fixed.txt", "w") as file:

    file.write("Class names: " + str(class_names) + "\n\n")

    file.write("Confusion matrix:\n")
    file.write(str(cm))
    file.write("\n\n")


    file.write("Classification report:\n")
    file.write(report)


print("Saved evaluation report to baseline_cnn_classification_report_fixed.txt")


# Transfer Learing With ResNet50 
Using a pretrained ResNet50 backbone with ImageNet weights, the backbone will be frozen. 

## Transfer Learning With ResNet50 Data Augmentation layer

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

resnet50_augmentation = keras.Sequential ([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1)
], name="resnet50_augmentation")

In [ ]:
# Pretrained resNet50 without org classification head
resnet50_backbone = keras.applications.ResNet50(
    weights = "imagenet",
    include_top = False,
    input_shape = (224,224, 3)
)

# Freeze the backbone of the pretrained model
resnet50_backbone.trainable = False

# Build the Transfer Learning model
resnet50_model =keras.Sequential([
    resnet50_augmentation, layers.Lambda(preprocess_input),resnet50_backbone, layers.GlobalAveragePooling2D(), layers.Dropout(0.2), layers.Dense(1, activation="sigmoid")
])

# Checking the structure of the model 
resnet50_model.summary()

## Compile ResNet 


In [ ]:
resnet50_model.compile(
    optimizer = "adam",
    loss= "binary_crossentropy",
    metrics = ["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall"), defective_recall]
)

## Train the model 

In [ ]:
EPOCHS_RESNET = 30

history_resnet = resnet50_model.fit(
    train_ds, 
    validation_data= val_ds,
    epochs = EPOCHS_RESNET
)

## Fine tune the last ResNet layers


In [ ]:
# Now we let the last part of ResNet learn a little bit
resnet50_backbone.trainable = True

for layer in resnet50_backbone.layers:
    layer.trainable = False

for layer in resnet50_backbone.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

resnet50_model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-5),
    loss = "binary_crossentropy",
    metrics = ["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall"), defective_recall]
)

EPOCHS_FINE_TUNE = 10

history_finetune = resnet50_model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = EPOCHS_FINE_TUNE
)


# Record Accuracy, loss, precision, recall and F1 Score for ResNet50 

In [ ]:
# Making a table of the training history
history_df = pd.DataFrame(history_resnet.history)
fine_tune_df = pd.DataFrame(history_finetune.history)
history_df = pd.concat([history_df, fine_tune_df], ignore_index=True)

# F1 Score combines precision and recall into one value
history_df["f1_score"] = 2 * (history_df["precision"] * history_df["recall"]) / (history_df["precision"] + history_df["recall"] + 1e-7)

# Validation F1 score uses validation precision and validation recall
history_df["val_f1_score"] = 2 * (history_df["val_precision"] * history_df["val_recall"]) / (history_df["val_precision"] + history_df["val_recall"] + 1e-7)

# Add an epoch number column so the table is easier to read 
history_df["epoch"] = range(1, len(history_df) +1)

# Choose the columns we want to record for the project report
metrics_table = history_df[[
    "epoch",
    "accuracy",
    "loss",
    "precision",
    "recall",
    "defective_recall",
    "f1_score",
    "val_accuracy",
    "val_loss",
    "val_precision",
    "val_recall",
    "val_defective_recall",
    "val_f1_score"
]]

# Save the metrics table as a CSV file
metrics_table.to_csv("resnet50_metrics_fixed.csv", index=False)

# Show the table below 
metrics_table


## Show and save the graphs of ResNet50 Results

In [ ]:
# One figure with 2 plots side by side
plt.figure(figsize=(12, 5))
resnet_accuracy = history_resnet.history["accuracy"] + history_finetune.history["accuracy"]
resnet_val_accuracy = history_resnet.history["val_accuracy"] + history_finetune.history["val_accuracy"]
resnet_loss = history_resnet.history["loss"] + history_finetune.history["loss"]
resnet_val_loss = history_resnet.history["val_loss"] + history_finetune.history["val_loss"]
resnet_epochs = range(1, len(resnet_accuracy) + 1)

# Plot 1: training accuracy vs validation accuracy
plt.subplot(1, 2, 1)
plt.plot(resnet_epochs, resnet_accuracy, label="Training accuracy")
plt.plot(resnet_epochs, resnet_val_accuracy, label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet50 Accuracy")
plt.legend()

# Plot 2: traing loss vs validation loss
plt.subplot(1, 2, 2)
plt.plot(resnet_epochs, resnet_loss, label="Training loss")
plt.plot(resnet_epochs, resnet_val_loss, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 Loss")
plt.legend()

# Make sure labels and titles don't overlap
plt.tight_layout()

# Save the plot for the report
plt.savefig("resnet50_accuracy_loss_fixed.png", dpi=300, bbox_inches="tight")

# Show the graph below

## Print the final record results 

In [ ]:
# Get the last row from the metrics table.
# The last row represents the final epoch of training.
final_results = metrics_table.iloc[-1]

# Print the final training and validation metrics in a readable format.
print("Final ResNet50 results")
print("Training accuracy:", final_results["accuracy"])
print("Training loss:", final_results["loss"])
print("Training precision:", final_results["precision"])
print("Training recall:", final_results["recall"])
print("Training F1-score:", final_results["f1_score"])
print("Training defective recall:", final_results["defective_recall"])
print("Validation accuracy:", final_results["val_accuracy"])
print("Validation loss:", final_results["val_loss"])
print("Validation precision:", final_results["val_precision"])
print("Validation recall:", final_results["val_recall"])
print("Validation F1-score:", final_results["val_f1_score"])
print("Validation defective recall:", final_results["val_defective_recall"])

## Confusion Matrix and Classifcation Report

In [ ]:
# y_true will store the true labels from the test data 
y_true = []

# y_pred will store the labels predicted by the trainde model 
y_pred = []

for images, labels in test_ds:

    probabilities = resnet50_model.predict(images)

    #convert predicted probabilites into 1D array with class labels 1 and 0 
    predicted_labels = (probabilities >= 0.5).astype("int32").flatten()

    #adding the real labels to y_true.
    y_true.extend(labels.numpy())

    #adding the predicted labels to y_pred.
    y_pred.extend(predicted_labels)


#Convert lists into numpy arrays so that sklearn can work with them easier 
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# This confirms that we have one prediction for each real label.
print("Class names:", class_names)
print("Number of real labels:", len(y_true))
print("Number of predicted labels:", len(y_pred))

#using the sklearn built in confusion matrix 
cm = confusion_matrix(y_true, y_pred)

# Create the classification report.
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
)

print("Class names:", class_names)

# Print the confusion matrix.
print(cm)

# Print the classification report.
print(report)


## Save the Evaluation Results of ResNet50 in a file 


In [ ]:
with open("resnet50_classification_report_fixed.txt", "w") as file:

    file.write("Class names: " + str(class_names) + "\n\n")

    file.write("Confusion matrix:\n")
    file.write(str(cm))
    file.write("\n\n")


    file.write("Classification report:\n")
    file.write(report)


print("Saved evaluation report to resnet50_classification_report_fixed.txt")